# Assignment 1 — Build a Custom Missing-Value Imputer
**Feature Engineering & MLOps · Unit 1, Session 4 follow-up (Missing Values)**

This notebook implements a scikit-learn-compatible `CustomImputer` class and applies it to the
PrepEdge `student_performance_raw.csv` dataset with strict train/test discipline.

## 0. Imports

In [1]:
import numpy as np
import pandas as pd
import pandas.api.types as ptypes

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
np.random.seed(42)

## 1. `CustomImputer` class

Design notes:
- Follows the scikit-learn estimator API (`BaseEstimator`, `TransformerMixin`) so it can be used
  inside `Pipeline`/`ColumnTransformer` just like `SimpleImputer`.
- `fit()` only ever *learns* statistics — it never fills anything in. `transform()` only ever *applies*
  the statistics that were learned during `fit()` — it never recomputes them. This is what enforces
  train-only fitting discipline once we call `.fit(X_train)` and `.transform(X_test)`.
- Column type (numeric vs. categorical) is auto-detected with `pandas.api.types.is_numeric_dtype`,
  so the user never has to specify it manually.
- `check_is_fitted` guards against calling `transform()` before `fit()`.
- A `column_overrides` dict (bonus) lets individual columns use a different strategy than the
  dataset-wide default.

In [2]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    A scikit-learn-compatible imputer for tabular data with mixed numeric and
    categorical columns.

    For every column, a fill value is learned during `fit()` (mean/median for
    numeric columns, the mode for categorical columns) and then applied to any
    data passed to `transform()`. Missing values are never imputed differently
    between train and test: `transform()` only ever reuses statistics that were
    already learned on the training data passed to `fit()`.

    Parameters
    ----------
    numeric_strategy : {'mean', 'median'}, default='median'
        Statistic used to fill missing values in numeric columns, unless
        overridden per-column via `column_overrides`.
    categorical_strategy : {'most_frequent'}, default='most_frequent'
        Strategy used to fill missing values in non-numeric columns, unless
        overridden per-column via `column_overrides`.
    add_missing_indicator : bool, default=True
        If True, `transform()` adds one extra binary column named
        `<column>_was_missing` for every column that contained at least one
        missing value in the TRAINING data (connects to the MCAR/MAR/MNAR
        discussion — the fact that a value was missing can itself be signal).
    column_overrides : dict or None, default=None
        Optional {column_name: strategy} mapping that overrides the
        dataset-wide strategy for specific columns (bonus feature).
        e.g. {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}

    Attributes
    ----------
    fill_values_ : dict
        {column_name: learned fill value}, set by `fit()`.
    numeric_cols_ : list
        Columns auto-detected as numeric during `fit()`.
    categorical_cols_ : list
        Columns auto-detected as categorical during `fit()`.
    missing_cols_ : list
        Columns that had at least one missing value in the training data
        (these are the columns that get a `_was_missing` indicator).
    """

    def __init__(self, numeric_strategy='median', categorical_strategy='most_frequent',
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _strategy_for(self, col, is_numeric):
        """Return the strategy to use for `col`, respecting column_overrides."""
        if self.column_overrides and col in self.column_overrides:
            return self.column_overrides[col]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    def fit(self, X, y=None):
        """
        Learn the fill value for every column of X, using X ONLY (this must
        be the training split — never the full dataset or the test split).

        Parameters
        ----------
        X : pandas.DataFrame of shape (n_samples, n_features)
        y : ignored
            Present for scikit-learn API compatibility.

        Returns
        -------
        self : CustomImputer
            Fitted imputer.
        """
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.numeric_cols_ = [c for c in X.columns if ptypes.is_numeric_dtype(X[c])]
        self.categorical_cols_ = [c for c in X.columns if c not in self.numeric_cols_]
        self.missing_cols_ = [c for c in X.columns if X[c].isna().any()]

        fill_values = {}
        for col in X.columns:
            is_numeric = col in self.numeric_cols_
            strategy = self._strategy_for(col, is_numeric)
            series = X[col].dropna()

            if is_numeric:
                if strategy == 'mean':
                    fill_values[col] = series.mean()
                elif strategy == 'median':
                    fill_values[col] = series.median()
                else:
                    raise ValueError(f"Unknown numeric strategy '{strategy}' for column '{col}'")
            else:
                if strategy == 'most_frequent':
                    mode = series.mode()
                    fill_values[col] = mode.iloc[0] if len(mode) else np.nan
                else:
                    raise ValueError(f"Unknown categorical strategy '{strategy}' for column '{col}'")

        self.fill_values_ = fill_values
        return self

    def transform(self, X):
        """
        Return a COPY of X with missing values filled using the statistics
        learned in `fit()` (nothing is recomputed here), plus optional
        `<col>_was_missing` indicator columns.

        Parameters
        ----------
        X : pandas.DataFrame of shape (n_samples, n_features)

        Returns
        -------
        pandas.DataFrame
            A new DataFrame — the original `X` is left untouched.
        """
        check_is_fitted(self, 'fill_values_')
        X = pd.DataFrame(X).copy()

        if self.add_missing_indicator:
            for col in self.missing_cols_:
                if col in X.columns:
                    X[f'{col}_was_missing'] = X[col].isna().astype(int)

        for col in X.columns:
            if col in self.fill_values_ and X[col].isna().any():
                X[col] = X[col].fillna(self.fill_values_[col])

        return X

## 2. Load the dataset

In [3]:
df = pd.read_csv('/Users/bhagyashreebhagat/Downloads/Assignments/raw/student_performance_raw.csv')
print(df.shape)
df.isna().sum()[df.isna().sum() > 0]

(600, 17)


weekly_study_hours    36
income_bracket        30
prev_exam_score       25
mock_test_3           21
feedback_text         68
dtype: int64

## 3. Train/test split (80/20)

We split BEFORE imputing anything. `student_id` is an identifier (not a feature) so it is set aside;
`final_score` is the regression target and is kept in `y` rather than imputed as a feature.

In [4]:
feature_cols = [c for c in df.columns if c not in ('student_id', 'final_score')]

X = df[feature_cols]
y = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

Train shape: (480, 15)  Test shape: (120, 15)


## 4. Fit on TRAIN only, transform both splits

In [5]:
imputer = CustomImputer(numeric_strategy='median', categorical_strategy='most_frequent',
                         add_missing_indicator=True)
imputer.fit(X_train)   # <-- fitted on the training split ONLY

X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print('Columns learned as numeric:', imputer.numeric_cols_)
print()
print('Columns with missing values in TRAIN (get indicator flags):', imputer.missing_cols_)
print()
print('New shape after adding indicator columns -> train:', X_train_imputed.shape,
      ' test:', X_test_imputed.shape)

Columns learned as numeric: ['city_tier', 'age', 'attendance_pct', 'weekly_study_hours', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended']

Columns with missing values in TRAIN (get indicator flags): ['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']

New shape after adding indicator columns -> train: (480, 20)  test: (120, 20)


## 5. Verify no missing values remain

In [6]:
original_cols = [c for c in imputer.columns_]  # excludes the *_was_missing indicator columns

train_missing_after = X_train_imputed[original_cols].isna().sum().sum()
test_missing_after = X_test_imputed[original_cols].isna().sum().sum()

print('Remaining missing values in TRAIN (imputed columns):', train_missing_after)
print('Remaining missing values in TEST  (imputed columns):', test_missing_after)

assert train_missing_after == 0
assert test_missing_after == 0
print('\nPASSED: no missing values remain in either split.')

Remaining missing values in TRAIN (imputed columns): 0
Remaining missing values in TEST  (imputed columns): 0

PASSED: no missing values remain in either split.


## 6. Mean / std before vs. after imputation (numeric columns)

In [7]:
numeric_missing_cols = [c for c in imputer.numeric_cols_ if c in imputer.missing_cols_]

rows = []
for col in numeric_missing_cols:
    before = X_train[col]
    after = X_train_imputed[col]
    rows.append({
        'column': col,
        'mean_before': before.mean(),
        'mean_after': after.mean(),
        'std_before': before.std(),
        'std_after': after.std(),
    })

summary = pd.DataFrame(rows).round(3)
summary

,column,mean_before,mean_after,std_before,std_after
0,weekly_study_hours,6.064,6.009,4.350,4.242
1,prev_exam_score,65.825,65.838,14.371,14.022
2,mock_test_3,70.702,70.720,19.077,18.777


**Comment:** the mean barely moves (median/mean imputation is designed to preserve the central
tendency), but the standard deviation shrinks a little for every column — we've replaced genuinely
variable missing entries with a single constant value, which mechanically reduces spread. The effect
is largest for whichever column had the most missing values.

## 7. Sanity check against `sklearn.impute.SimpleImputer`

In [8]:
sk_imputer = SimpleImputer(strategy='median')
sk_imputer.fit(X_train[numeric_missing_cols])

comparison = pd.DataFrame({
    'column': numeric_missing_cols,
    'CustomImputer_fill_value': [imputer.fill_values_[c] for c in numeric_missing_cols],
    'SimpleImputer_fill_value': sk_imputer.statistics_,
})
comparison['match'] = np.isclose(comparison['CustomImputer_fill_value'],
                                  comparison['SimpleImputer_fill_value'])
comparison

,column,CustomImputer_fill_value,SimpleImputer_fill_value,match
0,weekly_study_hours,5.0,5.0,True
1,prev_exam_score,66.1,66.1,True
2,mock_test_3,71.3,71.3,True


In [9]:
assert comparison['match'].all(), "CustomImputer fill values do not match SimpleImputer!"
print('PASSED: CustomImputer produces the same fill values as SimpleImputer.')

PASSED: CustomImputer produces the same fill values as SimpleImputer.


## 8. Reflection questions

**1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?**

Fitting on anything beyond the training split lets information about the test rows leak into the
statistics used to fill missing values — the classic form of *data leakage*. If the test set's values
influence the mean/median/mode, the model gets an unrealistically favorable preview of the test
distribution during training, and performance metrics measured later are no longer a fair estimate of
how the model will do on truly unseen data. Fitting on `X_train` only, and simply *applying* those
already-learned numbers to `X_test`, mirrors what happens in production: at prediction time you will
never have access to future/unseen data to recompute statistics from.

**2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem? What does `add_missing_indicator` contribute?**

No — mean/median imputation does not solve the underlying problem for an MNAR column. Because
`mock_test_3` is missing specifically *because* its true value would have been low (students who did
poorly skipped or didn't report the mock test), filling those slots with the overall median actively
biases the column upward and hides the very pattern that caused the missingness in the first place. The
`add_missing_indicator` flag doesn't fix the biased numeric value, but it does hand the model a second,
honest signal — "this student's mock_test_3 was originally missing" — which a downstream model can
learn to associate with the same underlying weak performance the raw value was masking.

**3. A brand-new column, entirely missing in training but present in test — what does the current implementation do, and what should a production version do instead?**

As written, `fit()` never sees that column, so it never gets an entry in `fill_values_`. In
`transform()`, the check `if col in self.fill_values_` evaluates to `False` for that column, so the
column is silently passed through **still full of missing values** — no error, no fill, which is a
silent failure that could crash a downstream model or corrupt predictions. A production-grade version
should instead (a) raise a clear warning/error the first time an unseen column appears (fail loudly
rather than silently), or (b) explicitly define a fallback policy for unseen columns — e.g. drop them,
or impute with a documented default (0 for numeric, a sentinel category like `"unknown"` for
categorical) — and log that decision so it's auditable rather than invisible.

## 9. Bonus — per-column strategy override (`column_overrides`)

In [10]:
bonus_imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    column_overrides={
        'weekly_study_hours': 'mean',       # numeric column, overridden to 'mean'
        'income_bracket': 'most_frequent',  # categorical column, explicit override (same as default)
    }
)
bonus_imputer.fit(X_train)

print("Default-strategy fill value for weekly_study_hours (median):",
      round(imputer.fill_values_['weekly_study_hours'], 3))
print("Override-strategy fill value for weekly_study_hours (mean):  ",
      round(bonus_imputer.fill_values_['weekly_study_hours'], 3))
print()
print("income_bracket fill value (most_frequent, both cases):",
      bonus_imputer.fill_values_['income_bracket'], '/', imputer.fill_values_['income_bracket'])

Default-strategy fill value for weekly_study_hours (median): 5.0
Override-strategy fill value for weekly_study_hours (mean):   6.064

income_bracket fill value (most_frequent, both cases): 5-10L / 5-10L


The two fill values for `weekly_study_hours` differ (mean vs. median), confirming that
`column_overrides` correctly lets a single column deviate from the dataset-wide default strategy,
while `income_bracket` — explicitly overridden to the same strategy it already used — is unchanged.